This notebook prepares the address information provided within NHSD.
The output is a single csv file, wide format, uniquely identified per ID.
The old version does the cleaning seperately from the clean_cohort_lsoa
New process start with only retaining IDs who were linked and has permission.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os




In [ ]:
df_lsoa = pd.read_stata(r"S:\LLC_0002\data\stata_w_labs\CORE_nhsd_lsoa11_v0002_20231212.dta")



In [ ]:
df_lsoa

In [ ]:
# load in cohort csv - repeat this!

df_cohort = pd.read_csv(r"S:\LLC_0002\lamj\notebooks_lsoa\clean_cohort_id.csv", low_memory=False)
df_cohort

In [ ]:
df = df_cohort[~df_cohort['llc_0002_stud_id'].isin(df_lsoa['llc_0002_stud_id'])]

In [ ]:
df

In [ ]:
df.drop_duplicates(subset='llc_0002_stud_id', keep = 'first')

In [ ]:
# only keep if ID in this file appears in cohort csv.
filtered_df = df_lsoa[df_lsoa['llc_0002_stud_id'].isin(df_cohort['llc_0002_stud_id'])]
df_lsoa = filtered_df

In [ ]:
df_lsoa

In [ ]:
df_lsoa['llc_0002_stud_id'].value_counts()

In [ ]:
df_lsoa[df_lsoa['origin'] == "DEMOGRAPHICS"].count()


In [ ]:
# only include source != demographics - given PDS is not yet available on LLC.
df_lsoa = df_lsoa[df_lsoa['origin'] != "DEMOGRAPHICS"]

In [ ]:
df_lsoa['llc_0002_stud_id'].count()

In [ ]:
# sort and count 
df_lsoa.sort_values(by = ["llc_0002_stud_id", 'record_date'], inplace = True)
df_lsoa['num_lsoa'] = df_lsoa.groupby('llc_0002_stud_id')['lsoa11cd_e'].transform('count')

In [ ]:
df_lsoa['has_lsoa'] = 0
df_lsoa.loc[(df_lsoa['num_lsoa'] != 0), 'has_lsoa'] = 1

filtered_df = df_lsoa[df_lsoa['has_lsoa'] == 0]
filtered_df = filtered_df.drop_duplicates(subset = 'llc_0002_stud_id', keep = 'first')
rows_to_update =  ~df_lsoa['llc_0002_stud_id'].isin(filtered_df['llc_0002_stud_id'])
df_lsoa.loc[rows_to_update, 'has_lsoa'] = 1



In [ ]:
# export a version to merge with demographics to describe IDs with and without LSOAs
selected_columns = ['llc_0002_stud_id', 'has_lsoa', 'origin']
df_nolsoa = filtered_df[selected_columns]
df_nolsoa.rename(columns = {"llc_0002_stud_id": "LLC_0002_stud_id"})


# 172 IDs without lsoas

In [ ]:
df_nolsoa['source'] = "no NHSD lsoa"
df['source'] = "not in NHSD"
combined_df = pd.concat([df, df_nolsoa], ignore_index = True)


In [ ]:
combined_df

In [ ]:
combined_df.to_csv("nolsoa_ids.csv")


In [ ]:
df_lsoa[df_lsoa['has_lsoa'] == 0]

In [ ]:
df_lsoa = df_lsoa[df_lsoa['has_lsoa'] != 0]

In [ ]:
# re-format date
df_lsoa['date'] = df_lsoa['record_date'].str[:10]
df_lsoa['date'] = pd.to_datetime(df_lsoa['date'], format = '%Y-%m-%d')

In [ ]:
# clean date - missing, 1800-01-01, 1860-01-01, 1899-12-30 - GDPPR, before 1920...?
# HES APC started in April 1989
# Recode dates before 1989 April as Missing/NaT (<0.01%)

# recode above to missing 
df_lsoa.loc[(df_lsoa['date'].dt.year<1988), 'date'] = pd.NaT
df_lsoa.loc[(df_lsoa['date'].dt.year<1989) & (df_lsoa['date'].dt.month < 4), 'date'] = pd.NaT
missing_date_count = df_lsoa['date'].isnull().sum()

# define the last available date for each address as end_date
df_lsoa['end_date'] = df_lsoa.groupby('llc_0002_stud_id')['date'].transform('last')

In [ ]:
missing_date_count

In [ ]:
df_lsoa

In [ ]:
# check if for invalid dates
((df_lsoa['date'] <= "1970-01-01") == 1).sum()

In [ ]:
df_lsoa['date'].isna().sum()

In [ ]:
df_lsoa = df_lsoa.dropna(subset = ['date'])

In [ ]:
df_lsoa

In [ ]:
# DROP if no lsoa (create sub-set of IDs with 1 or more LSOAs)
df_lsoa = df_lsoa.dropna(subset = ['lsoa11cd_e'])

In [ ]:
df_lsoa

In [ ]:
df_lsoa.groupby('llc_0002_stud_id').count()

In [ ]:
df_lsoa['origin'].value_counts()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import datetime

plt.figure(figsize= (10,6))
plt.hist(df_lsoa['date'], bins = 200, edgecolor = 'black')
plt.xlim(10000, 20000)
plt.show()

In [ ]:
# sort and count again. 
df_lsoa.sort_values(by = ["llc_0002_stud_id", 'date'], inplace = True)
df_lsoa['num_lsoa'] = df_lsoa.groupby('llc_0002_stud_id')['lsoa11cd_e'].transform('count')
df_lsoa['has_lsoa'] = 0
df_lsoa.loc[(df_lsoa['num_lsoa'] != 0), 'has_lsoa'] = 1

In [ ]:
df_lsoa

In [ ]:
# sort and count
df_lsoa.sort_values(by = ["llc_0002_stud_id", 'date'], inplace = True)
df_lsoa['num_lsoa'] = df_lsoa.groupby('llc_0002_stud_id')['lsoa11cd_e'].transform('count')


In [ ]:
# count lsoa changes
df_lsoa['lsoa_change'] = (df_lsoa['lsoa11cd_e'] != df_lsoa['lsoa11cd_e'].shift()) & (df_lsoa.groupby('llc_0002_stud_id').cumcount() > 0)

# code the first (non-missing) address as "True" such that they are kept.
df_lsoa.loc[df_lsoa.groupby('llc_0002_stud_id').head(1).index, 'lsoa_change'] = True

In [ ]:
# Sum - total changes (minus 1 since we coded the first row as a change - in order not to drop it)
df_lsoa['lsoa_change_count'] = df_lsoa.groupby('llc_0002_stud_id')['lsoa_change'].transform('sum') - 1
df_lsoa['total_lsoa_per_id'] = df_lsoa['lsoa_change_count'] + 1 

In [ ]:
# describe population with only 1 lsoa
one_lsoa = df_lsoa[df_lsoa['total_lsoa_per_id'] == 1] 
one_lsoa = one_lsoa.groupby('llc_0002_stud_id').head(1)
one_lsoa


In [ ]:
one_lsoa.to_csv('one_LSOA_NHSD.csv')

In [ ]:
# calculate total number of changes
unique_change_counts = df_lsoa.groupby('llc_0002_stud_id')['lsoa_change_count'].head(1)
unique_change_counts.describe()
#unique_change_counts.sum()

In [ ]:
unique_change_counts[(unique_change_counts >= 15) & (unique_change_counts < 20) ].count()


In [ ]:
df_lsoa['lsoa11cd_e'].count()

In [ ]:
df_lsoa

In [ ]:
# histogram for number of changes 
plt.figure(figsize= (10,6))
plt.hist(unique_change_counts, bins = 200, edgecolor = 'black')
plt.xlim(0, 100)
plt.show()

In [ ]:
# count unique lsoa - only relevant for changers
changer = df_lsoa[df_lsoa['total_lsoa_per_id'] != 1] 
unique_lsoa_counts = changer.groupby('llc_0002_stud_id')['lsoa11cd_e'].nunique()
unique_lsoa_counts.describe()

In [ ]:
unique_lsoa_counts.sum()

In [ ]:
df_lsoa['lsoa11cd_e'].value_counts()

In [ ]:
df_lsoa.sort_values(by = ["llc_0002_stud_id", 'date'], inplace = True)


In [ ]:
df_lsoa.head(5)


In [ ]:
# create unique id dataset to understand distribution  
df_unique = df_lsoa.drop_duplicates(subset='llc_0002_stud_id', keep = 'first')

In [ ]:
columns_to_keep = ["llc_0002_stud_id", "total_lsoa_per_id", "num_lsoa"]
df_unique = df_unique[columns_to_keep]
df_unique['total_lsoa_per_id'] = pd.to_numeric(df_unique['total_lsoa_per_id'], errors = 'coerce')
df_unique['total_lsoa_per_id'].dtype

In [ ]:
df_unique

In [ ]:
# df_unique.loc[df_unique['total_lsoa_per_id'] >= 2, 'total_lsoa_per_id'] = 2
df_unique = df_unique.rename(columns = {'total_lsoa_per_id': 'num_different_lsoa'})

In [ ]:
df_unique.to_csv("ID_to_link_with_demographics.csv")

In [ ]:
df_unique['change_count'] = df_unique['num_different_lsoa'] - 1


In [ ]:
df_unique[(df_unique['change_count']>=15)].count() 
          # & (df_unique['change_count']<= 14)].count()
        

In [ ]:
# keep only the changes (drop rows where LSOA doesn't change)
# this is beacuse update of records without changes is still kept here
df_lsoa.sort_values(by = ['llc_0002_stud_id','date'], inplace = True)
df_lsoa = df_lsoa[df_lsoa['lsoa_change']] ## since we flagged all first rows as changes, they are kept here.


In [ ]:
# formatting variables
# encrypted lsoa = str
# date = str

df_lsoa['lsoa11cd_e'] = df_lsoa['lsoa11cd_e'].astype(int)
df_lsoa['lsoa11cd_e'] = df_lsoa['lsoa11cd_e'].astype(str)


In [ ]:
# add step: converting to json file.
# datetime to object as json cannot work with datetime format
# if save to csv, date == obj
import numpy as np

df_lsoa['date'] = np.datetime_as_string(df_lsoa['date'], unit = 'D')

In [ ]:
df_lsoa[df_lsoa['date'].isna()]

In [ ]:
# ID = str
df_lsoa['llc_0002_stud_id'] = df_lsoa['llc_0002_stud_id'].astype(str)

In [ ]:
df_lsoa['llc_0002_stud_id'] = df_lsoa['llc_0002_stud_id'].str.replace('.0', '',regex= False)


In [ ]:
# restructure data to dictionary

from collections import defaultdict

result = defaultdict(lambda: {'origin': [], 'lsoa11cd_e':[],'start_date':[],'total_addresses_nhsd':[]})



In [ ]:
df_lsoa['lsoa11cd_e'].describe()

In [ ]:
for _, row in df_lsoa.iterrows():
    llc_0002_stud_id = row['llc_0002_stud_id']
    result[llc_0002_stud_id]['total_addresses_nhsd'] = row['total_lsoa_per_id']
    result[llc_0002_stud_id]['origin'].append(row['origin'])
    result[llc_0002_stud_id]['lsoa11cd_e'].append(row['lsoa11cd_e'])
    result[llc_0002_stud_id]['start_date'].append(row['date'])

In [ ]:
result

In [ ]:
# save dictionary - json
import json
with open("nhsd_geo.json", "w") as outfile:
    json.dump(result, outfile)
    

